# Demo Setup

This notebook creates the schema and tables used in **Demo: Window Functions and CTEs in Databricks SQL**.

In [0]:
# Get current user and set up schema name
import re

username = spark.sql("SELECT current_user()").first()[0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog_name = spark.sql("SELECT current_catalog()").first()[0]
schema_name = f"demo_windows_cte_{clean_username}"

# Create the schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}` COMMENT 'Demo schema for window functions and CTEs'")
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"USE SCHEMA `{schema_name}`")

In [0]:
# Create daily_sales table - 90 days of daily revenue by region
# Used for sliding window / rolling average demos
import random
from pyspark.sql import Row
from datetime import date, timedelta

random.seed(77)

regions = ['North', 'South', 'East', 'West']
start_date = date(2024, 1, 1)

rows = []
for day_offset in range(90):  # Jan 1 - Mar 30, 2024
    d = start_date + timedelta(days=day_offset)
    for region in regions:
        # Base revenue varies by region, with day-of-week seasonality and noise
        base = {'North': 12000, 'South': 10500, 'East': 14000, 'West': 13000}[region]
        # Weekend dip
        dow_factor = 0.7 if d.weekday() >= 5 else 1.0
        # Random variation
        revenue = base * dow_factor * random.uniform(0.8, 1.2)
        orders = int(revenue / random.uniform(55, 85))
        rows.append(Row(
            sale_date=d,
            region=region,
            daily_revenue=round(revenue, 2),
            order_count=orders
        ))

df_daily = spark.createDataFrame(rows)
df_daily.write.mode("overwrite").saveAsTable("daily_sales")

count = spark.sql("SELECT COUNT(*) FROM daily_sales").first()[0]

In [0]:
# Create sensor_readings table - IoT data with intentional duplicates
# Used for deduplication demos (ROW_NUMBER + QUALIFY)
import random
from pyspark.sql import Row
from datetime import datetime, timedelta

random.seed(42)

sensor_ids = ['sensor_A', 'sensor_B', 'sensor_C', 'sensor_D']
base_time = datetime(2024, 3, 15, 8, 0, 0)

rows = []
reading_id = 1
for sensor in sensor_ids:
    for i in range(20):  # 20 readings per sensor
        ts = base_time + timedelta(minutes=i * 15)  # every 15 minutes
        temperature = round(random.uniform(18.0, 26.0), 1)
        humidity = round(random.uniform(35.0, 65.0), 1)
        
        # Normal reading
        rows.append(Row(
            reading_id=reading_id,
            sensor_id=sensor,
            reading_timestamp=ts,
            temperature=temperature,
            humidity=humidity
        ))
        reading_id += 1
        
        # ~25% chance of a duplicate (same sensor, same timestamp, slightly different reading_id)
        if random.random() < 0.25:
            rows.append(Row(
                reading_id=reading_id,
                sensor_id=sensor,
                reading_timestamp=ts,  # same timestamp = duplicate!
                temperature=temperature,
                humidity=humidity
            ))
            reading_id += 1

df_sensors = spark.createDataFrame(rows)
df_sensors.write.mode("overwrite").saveAsTable("sensor_readings")

total = spark.sql("SELECT COUNT(*) FROM sensor_readings").first()[0]
dupes = total - spark.sql("SELECT COUNT(DISTINCT sensor_id, reading_timestamp) FROM sensor_readings").first()[0]

In [0]:
print("SETUP COMPLETE")
print(f"")
print(f"Catalog:  {catalog_name}")
print(f"Schema:   {schema_name}")
print(f"")
print(f"Tables created:")
print(f"daily_sales       - 90 days of daily revenue (sliding window demos)")
print(f"sensor_readings   - IoT readings with duplicates (deduplication demos)")